In [5]:
# csv
# Outlook,Temperature,Humidity,Wind,Play
# Sunny,Hot,High,Weak,No
# Sunny,Hot,High,Strong,No
# Overcast,Hot,High,Weak,Yes
# Rain,Mild,High,Weak,Yes
# Rain,Cool,Normal,Weak,Yes
# Rain,Cool,Normal,Strong,No
# Overcast,Cool,Normal,Strong,Yes
# Sunny,Mild,High,Weak,No
# Sunny,Cool,Normal,Weak,Yes
# Rain,Mild,Normal,Weak,Yes
# Sunny,Mild,Normal,Strong,Yes
# Overcast,Mild,High,Strong,Yes
# Overcast,Hot,Normal,Weak,Yes
# Rain,Mild,High,Strong,No


import pandas as pd
import math

# Load dataset
data = pd.read_csv("dt.csv")


# Calculate Entropy
def entropy(data):
    labels = data.iloc[:, -1].value_counts()
    total = len(data)

    return sum(
        -(c / total) * math.log2(c / total)
        for c in labels
    )


# Calculate Information Gain
def info_gain(data, col):
    total_entropy = entropy(data)

    weighted = sum(
        (len(data[data[col] == v]) / len(data)) *
        entropy(data[data[col] == v])
        for v in data[col].unique()
    )

    return total_entropy - weighted


# Build Decision Tree
def build_tree(data, features):
    labels = data.iloc[:, -1]

    # If all outputs are same
    if len(labels.unique()) == 1:
        return labels.iloc[0]

    # If no features left
    if not features:
        return labels.mode()[0]

    # Choose best feature
    best = max(features, key=lambda f: info_gain(data, f))

    tree = {best: {}}

    # Create branches
    for val in data[best].unique():
        subset = data[data[best] == val].drop(columns=[best])

        tree[best][val] = build_tree(
            subset,
            [f for f in features if f != best]
        )

    return tree


# Classify new sample
def classify(tree, sample):

    # If leaf node
    if not isinstance(tree, dict):
        return tree

    feat = list(tree.keys())[0]
    val = sample.get(feat)

    if val in tree[feat]:
        return classify(tree[feat][val], sample)

    return "Unknown"


# Build tree
tree = build_tree(data, list(data.columns[:-1]))

print("Tree:")
print(tree)

# Test sample
sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'Normal',
    'Wind': 'Strong'
}

print("Prediction:", classify(tree, sample))


# output:
# Tree:
# {'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}
# Prediction: Yes

Tree:
{'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}
Prediction: Yes
